# Exploratory Data Auditing: A Typology of Data Anomalies
[Brian C. Keegan, Ph.D.](http://www.brianckeegan.com)  
Heather Carrasco (University of New Mexico)  
May 2026

Released under an [MIT License](https://opensource.org/licenses/MIT).

This notebook is the analytic companion to *Exploratory Data Auditing: A Typology of Data Anomalies Using U.S. House of Representatives Disbursements* (submitted to *Harvard Data Science Review*). Section headings here mirror the manuscript's section structure so that every figure and pinned number in the paper can be located by section. The notebook consumes the single cleaned artifact produced by [`cleaning.ipynb`](cleaning.ipynb) (`all_disbursements.csv`, Git-LFS-tracked) and performs no schema normalisation or datetime parsing of its own — that is the cleaning notebook's contract. Analysis-specific transforms (personnel payee-name normalisation, gender/party/office enrichment, per-test statistical subsetting) remain here by design.


## Setup

### Imports and environment

In [1]:
import numpy as np
import pandas as pd

pd.options.display.max_columns = 100
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
idx = pd.IndexSlice

%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sb

from scipy import stats
import networkx as nx
from geopy.distance import geodesic


### Publication-ready matplotlib defaults

These `rcParams` apply to every figure produced downstream. A single block here means the manuscript figures and any inline rendering share the same typography, weight, colour, and tick formatting — no figure inherits matplotlib's defaults. The `PALETTE` dictionary holds every named colour used in the notebook so individual cells never hard-code hex values.

In [2]:
# Publication-ready defaults. Applied globally via rcParams so individual
# figure cells stay short and consistent.
mpl.rcParams.update({
    # Figure & save
    'figure.dpi':            110,
    'figure.figsize':        (7.0, 4.3),
    'figure.facecolor':      'white',
    'savefig.dpi':           300,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.format':        'png',

    # Typography
    'font.family':           'serif',
    'font.serif':            ['DejaVu Serif', 'Bitstream Vera Serif', 'serif'],
    'font.size':             11,
    'axes.titlesize':        12,
    'axes.titleweight':      'semibold',
    'axes.labelsize':        11,
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 10,

    # Spines, axes
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.9,
    'axes.edgecolor':        '#222222',
    'axes.labelcolor':       '#222222',
    'axes.titlecolor':       '#222222',
    'axes.axisbelow':        True,

    # Ticks
    'xtick.color':           '#222222',
    'ytick.color':           '#222222',
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.major.width':     0.9,
    'ytick.major.width':     0.9,
    'xtick.minor.size':      2,
    'ytick.minor.size':      2,

    # Grid (off by default; enabled per-axis where useful)
    'grid.color':            '#dddddd',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.5,
    'axes.grid':             False,

    # Lines & markers
    'lines.linewidth':       2.2,
    'lines.markeredgewidth': 0.5,
    'lines.solid_capstyle':  'round',

    # Patches (bars)
    'patch.linewidth':       0.4,
    'patch.edgecolor':       'white',

    # Legend
    'legend.frameon':        False,
    'legend.borderaxespad':  0.5,
})

# Project palette — every named colour in one place
PALETTE = {
    'positive':   '#1F4E79',  # positive disbursements (deep blue)
    'negative':   '#A6332E',  # reimbursements (deep red)
    'neutral':    '#6B6B6B',  # references / medians
    'highlight':  '#A6332E',  # the anomaly being called out
    'benford':    '#000000',  # Benford reference line
    'bar':        '#4A7CB3',  # neutral observed-data bar
    'bar_alt':    '#D67D2C',  # secondary bar (multiples of 25 in Fig 2)
    'bar_warn':   '#A6332E',  # tertiary bar (multiples of 10 in Fig 2)
    'band':       '#9EC5E8',  # IQR ribbon
    'D':          '#2E5A87',  # Democratic
    'R':          '#A6332E',  # Republican
    'B':          '#7B5BA6',  # bipartisan
    'F':          '#D67D2C',  # female
    'M':          '#3B7A8A',  # male
    'commercial': '#1F4E79',
    'car':        '#3B7A8A',
    'lodging':    '#D67D2C',
}

# Periodic-cycle markers used across temporal figures
CYCLES = [
    ('2 weeks',   14,    '#A6332E'),
    ('1 month',   30,    '#2E5A87'),
    ('1 quarter', 91,    '#7B5BA6'),
    ('2 years',   365*2, '#5C4033'),
]


### Top expense categories

Defined once at the top of the notebook so categorical sub-analyses and figure subsetting use the same eight-category universe consistently.

In [3]:
top_cats = [
    'PERSONNEL COMPENSATION',
    'FRANKED MAIL',
    'TRAVEL',
    'RENT COMMUNICATION UTILITIES',
    'PRINTING AND REPRODUCTION',
    'OTHER SERVICES',
    'SUPPLIES AND MATERIALS',
    'EQUIPMENT',
]


## Data loading and preparation

*Maps to manuscript §Background → U.S. House of Representatives expenditures.*

The disbursement universe and four enrichment tables are loaded once at the top of the notebook. Every dataset is named to its canonical source and validated with a printed assertion before any downstream cell uses it.

### Statement of Disbursements

**Goal.** Load the cleaned, schema-normalised member disbursement universe (2011 Q1 – 2022 Q4) and verify the integration contract with `cleaning.ipynb`. This is the dataset that produces every analysis in the manuscript.

**Assumptions.** `cleaning.ipynb` has already standardised column names, parsed dates, filtered out 2009–2010 (incomplete reporting), and produced `all_disbursements.csv` as a Git-LFS-tracked file colocated with this notebook. This notebook does not re-do that work.

**Interpretation.** Each row is one disbursement (a positive payment or a negative reimbursement). `AMOUNT` is in USD. `BIOGUIDE_ID` is the canonical congressional identifier; non-member offices (Speaker, committees) have null `BIOGUIDE_ID`.

**Validation.** The cell asserts the exact column set, datetime types on the four date columns, and that the cleaned universe starts in 2011. Any contract drift triggers a loud failure here rather than a silent corruption downstream.


In [4]:
# Cleaned, schema-stable artifact from cleaning.ipynb (Git-LFS-tracked).
all_db_df = pd.read_csv(
    'all_disbursements.csv',
    encoding='utf8',
    parse_dates=['DATE', 'PERIOD_DATE', 'START DATE', 'END DATE'],
    low_memory=False,
)
all_db_df.head()


,YEAR-QUARTER,YEAR,QUARTER,TERM_QUARTER,BIOGUIDE_ID,OFFICE,PROGRAM,CATEGORY,PAYEE,PURPOSE,AMOUNT,DATE,PERIOD_DATE,DATE_IS_RECONSTRUCTED,START DATE,END DATE,TRANSCODE,RECORDID,VOUCHER_ID
0,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,NaN,PERSONNEL COMPENSATION,"CASSIDY, ED",DIRECTOR OF HOUSE OPERATIONS,41066.67,NaT,2011-01-03,True,2011-01-03,2011-03-31,NaN,NaN,"CASSIDY, ED"
1,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,NaN,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR,958.33,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
2,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,NaN,PERSONNEL COMPENSATION,"DALY, BRENDAN",COMMUNICATIONS DIRECTOR (OTHER COMPENSATION),13416.67,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"DALY, BRENDAN"
3,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,NaN,PERSONNEL COMPENSATION,"ELSHAMI,NADEEM",DEPUTY COMMUNICATIONS DIRECTOR,943.73,NaT,2011-01-01,True,2011-01-01,2011-01-02,NaN,NaN,"ELSHAMI,NADEEM"
4,2011Q1,2011,Q1,1,NaN,OFFICE OF THE SPEAKER,NaN,PERSONNEL COMPENSATION,"GREEN, JO-MARIE S",GEN COUNSEL & CHIEF OF LEG OPS,42044.44,NaT,2011-01-03,True,2011-01-03,2011-03-31,NaN,NaN,"GREEN, JO-MARIE S"


Integration contract — fail loudly if the cleaning notebook's output schema drifts.

In [5]:
EXPECTED_COLUMNS = [
    'YEAR-QUARTER', 'YEAR', 'QUARTER', 'TERM_QUARTER', 'BIOGUIDE_ID', 'OFFICE',
    'PROGRAM', 'CATEGORY', 'PAYEE', 'PURPOSE', 'AMOUNT', 'DATE', 'PERIOD_DATE',
    'DATE_IS_RECONSTRUCTED', 'START DATE', 'END DATE', 'TRANSCODE', 'RECORDID',
    'VOUCHER_ID',
]
assert list(all_db_df.columns) == EXPECTED_COLUMNS, all_db_df.columns.tolist()
for _c in ['DATE', 'PERIOD_DATE', 'START DATE', 'END DATE']:
    assert pd.api.types.is_datetime64_any_dtype(all_db_df[_c]), _c
assert all_db_df['YEAR'].min() == 2011, 'cleaned universe must start 2011'

print(f'contract OK: {len(all_db_df):,} rows x '
      f'{len(all_db_df.columns)} cols, '
      f'{all_db_df["YEAR-QUARTER"].nunique()} quarters')


contract OK: 6,040,756 rows x 19 cols, 58 quarters


Filter to members' offices only. Disbursements by the Speaker's office, committees, and other non-member offices have null `BIOGUIDE_ID` and are excluded from every member-level analysis below.

In [6]:
members_df = all_db_df[all_db_df['BIOGUIDE_ID'].notnull()]
assert members_df['YEAR'].between(2011, 2022).all(), \
    'members_df must be 2011-2022'

print(f'{len(members_df):,} member-office disbursements '
      f'across {members_df["BIOGUIDE_ID"].nunique()} unique members')


3,843,881 member-office disbursements across 938 unique members


### Member biographical and party metadata

**Goal.** Attach gender, party, and full-name attributes to each member so categorical and relational analyses can stratify by demographic and partisan group.

**Assumptions.** `propublica_members.csv` was retrieved from ProPublica's [Congress API](https://projects.propublica.org/api-docs/congress-api/) — now deprecated; the file is committed to the repo as a snapshot.

**Interpretation.** The two helper dicts (`bioguide_gender_party_map`, `member_names_map`) are keyed on Bioguide ID. The notebook uses them for the gender/party categorical comparisons and for labelling members in the relational network.

**Validation.** `drop_duplicates(subset=['id'])` ensures one row per Bioguide ID; downstream membership lookups raise a `KeyError` if a member is missing rather than silently substituting NaN.


In [7]:
propublica_members_df = pd.read_csv('propublica_members.csv')
propublica_members_df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'propublica_members.csv'

In [ ]:
bioguide_gender_party_map = (
    propublica_members_df[['bioguide_id', 'gender', 'party']]
    .drop_duplicates(subset=['bioguide_id'])
    .set_index('bioguide_id')
    .to_dict('index')
)

member_names_map = propublica_members_df.copy()[['bioguide_id', 'first_name', 'last_name']]
member_names_map['full_name'] = (
    member_names_map['first_name'] + ' ' + member_names_map['last_name']
)
member_names_map = (
    member_names_map.drop_duplicates(subset=['bioguide_id'])
                     .set_index('bioguide_id')['full_name']
                     .to_dict()
)

print(f'{len(bioguide_gender_party_map):,} members with gender/party metadata')


### House Clerk office assignments

**Goal.** Attach office-building and office-room metadata (`CHOB-1`, `LHOB-2`, ...) so spatial-anomaly analyses can compare offices on the same floor of the same building.

**Assumptions.** `member_data_2015_2023.csv` is a flattening of the House Clerk's `MemberData.xml` snapshots harvested via the [Wayback Machine](https://web.archive.org/web/2015*/clerk.house.gov). Coverage starts 2015 because earlier `MemberData.xml` snapshots are either missing or have incompatible schemas.

**Interpretation.** `office-floor` is the first or second digit of the room number, depending on building convention (CHOB uses the first digit; LHOB and RHOB use the second). The dtype-stability assignment is needed because pandas 2.x refuses to overwrite a float64 NaN column with strings.

**Validation.** `head()` inspection confirms `office-floor` carries the expected `CHOB-N` / `LHOB-N` / `RHOB-N` strings.


In [ ]:
member_data_df = pd.read_csv('member_data_2015_2023.csv', encoding='utf8')
member_office_df = member_data_df.copy()[
    ['year', 'bioguideID', 'office-building', 'office-room']
]

# Initialise as object dtype so string assignments don't trip pandas 2.x's
# strict dtype check.
member_office_df['office-floor'] = pd.Series(
    pd.NA, index=member_office_df.index, dtype='object'
)

# CHOB rooms encode floor in the first digit (e.g. room 1234 → floor 1)
_chob = member_office_df['office-building'] == 'CHOB'
member_office_df.loc[_chob, 'office-floor'] = (
    'CHOB-' + member_office_df.loc[_chob, 'office-room'].astype(str).str.get(0)
)

# LHOB and RHOB encode floor in the second digit (e.g. room 1234 → floor 2)
_lhob = member_office_df['office-building'] == 'LHOB'
member_office_df.loc[_lhob, 'office-floor'] = (
    'LHOB-' + member_office_df.loc[_lhob, 'office-room'].astype(str).str.get(1)
)
_rhob = member_office_df['office-building'] == 'RHOB'
member_office_df.loc[_rhob, 'office-floor'] = (
    'RHOB-' + member_office_df.loc[_rhob, 'office-room'].astype(str).str.get(1)
)

member_office_df.head()


### Census Centers of Population and distance from D.C.

**Goal.** Build a state/territory reference table with population, centroid lat/long, and geodesic distance from the U.S. Capitol. This drives the spatial-anomaly analysis of travel spending vs. distance.

**Assumptions.** State centroids come from the Census Bureau's [2020 Centers of Population](https://www.census.gov/geographies/reference-files/time-series/geo/centers-population.html) (population-weighted, not geometric). U.S. territories are absent from that file and are filled by hand using 2020 Census Island Areas counts and populated-area centroids. U.S. Minor Outlying Islands has no House delegation and is dropped.

**Interpretation.** `DC_DIST` is geodesic kilometres from the Capitol (cop_df row 8 is Washington, D.C.). Used as the independent variable in the linear travel-vs-distance model.

**Validation.** Final `cop_df` has 56 rows (50 states + DC + 4 territories with delegations + 1 ANSI placeholder); UM has been dropped explicitly. `DC_DIST` is non-negative for every row and zero for D.C. itself.


In [ ]:
state_codes_df = pd.read_csv(
    'census_state.txt', sep='|', dtype={'STATE': str}
)
cop_df = pd.read_csv('census_cenpop2020.csv', dtype={'STATEFP': str})

cop_df = pd.merge(
    left=cop_df,
    right=state_codes_df,
    left_on='STATEFP', right_on='STATE',
    how='outer',
)

# Backfill the five territory rows that have no 2020 Center of Population
# record. Coordinates anchor on the populated area (Pago Pago, Saipan, etc.)
# rather than the geometric centroid; populations are 2020 Island Areas Census.
_terr = [
    # (df_idx, STATEFP, lat,    lon,     population)
    (51,      '60',    -14.27, -170.70,    49_710),  # American Samoa
    (52,      '66',     13.44,  144.79,   153_836),  # Guam
    (53,      '69',     15.18,  145.76,    47_329),  # Northern Mariana Islands
    (54,      '72',     18.22,  -66.59, 3_285_874),  # Puerto Rico
    (56,      '78',     18.34,  -64.93,    87_146),  # U.S. Virgin Islands
]
for _i, _fips, _lat, _lon, _pop in _terr:
    cop_df.loc[_i, ['STATEFP', 'LATITUDE', 'LONGITUDE', 'POPULATION']] = (
        _fips, _lat, _lon, _pop
    )

# U.S. Minor Outlying Islands (STUSAB='UM') has no House delegation and no
# meaningful single centroid (islands span Pacific + Caribbean). Drop it.
cop_df.drop(55, inplace=True)

# Geodesic distance from each centroid to D.C. (cop_df row 8 = Washington, DC)
_dc_lat, _dc_lon = cop_df.loc[8, 'LATITUDE'], cop_df.loc[8, 'LONGITUDE']
cop_df['DC_DIST'] = cop_df[['LATITUDE', 'LONGITUDE']].apply(
    lambda r: geodesic((r['LATITUDE'], r['LONGITUDE']), (_dc_lat, _dc_lon)).km,
    axis=1,
)

cop_df = cop_df[['STATEFP', 'STUSAB', 'STATE_NAME', 'POPULATION',
                 'LATITUDE', 'LONGITUDE', 'DC_DIST']]

assert (cop_df['DC_DIST'] >= 0).all()
assert cop_df['DC_DIST'].min() == 0, 'D.C. row should be zero distance from itself'
cop_df


### Joined member–disbursement table

**Goal.** Produce `members_bioguide_df`, the left-joined table that carries member demographics on every disbursement row. Used by gender/party comparisons.

**Assumptions.** A left join on `BIOGUIDE_ID` is appropriate because every row in `members_df` already has a non-null `BIOGUIDE_ID`. Members with no ProPublica metadata (none, in this snapshot) would surface as null `gender`/`party` and are filtered explicitly in downstream tests.

**Interpretation.** Use `members_bioguide_df` whenever a per-disbursement analysis needs demographic context. Use `members_df` for analyses keyed only on Bioguide ID, expense category, or amount.

**Validation.** Row count after join equals row count before join — no fan-out.


In [ ]:
members_bioguide_df = pd.merge(
    left=members_df,
    right=propublica_members_df,
    left_on='BIOGUIDE_ID',
    right_on='bioguide_id',
    how='left',
)

assert len(members_bioguide_df) == len(members_df), 'join caused row fan-out'
members_bioguide_df.head()


## Continuous anomalies

*Maps to manuscript §Exploratory Data Auditing → Continuous anomalies.*

Five analyses in this section: the personnel-compensation outlier survey (Figure 1), pre- and post-decimal digit distributions (Figure 2), Benford's law across categories (Figure 3) and at the member level for travel and personnel (Figure 4 + supplementary), and the relative size factor (Figure 5).

### Distribution of personnel compensation (Figure 1)

**Goal.** Show the distribution of personnel-compensation amounts across the entire 2011–2022 universe, pinning the central tendencies cited in the manuscript and visualising the long tail of high-value outliers.

**Assumptions.** Personnel compensation includes both positive disbursements (regular monthly or quarterly salary payments) and negative reimbursements (corrections, separations). Both are kept in the distribution.

**Interpretation.** Histogram bins are 5,000 dollars wide, anchored so zero is a bin edge. Blue bins are non-negative; red bins are non-positive. The log y-axis lets the modal salary range coexist with the rare five-figure outliers in one panel.

**Validation.** The cell asserts that zero falls on a bin edge — otherwise a bin straddling the sign boundary would be ambiguously coloured. The pinned headline numbers (mean, median, count of >$50k disbursements) are printed so the manuscript figures can be traced back to this cell.


In [ ]:
_s = members_df.loc[members_df['CATEGORY'] == 'PERSONNEL COMPENSATION', 'AMOUNT']

# Pinned headline numbers (manuscript text)
personnel_mean   = _s.mean()
personnel_median = _s.median()
personnel_gt20k_n = int((_s > 20_000).sum())
personnel_gt20k_pct = (_s > 20_000).mean()
personnel_gt50k_n = int((_s > 50_000).sum())
print(f'N personnel disbursements:       {len(_s):,}')
print(f'Mean amount:                     ${personnel_mean:,.2f}')
print(f'Median amount:                   ${personnel_median:,.2f}')
print(f'Disbursements > $20,000:         {personnel_gt20k_n:,} '
      f'({personnel_gt20k_pct:.1%})')
print(f'Disbursements > $50,000:         {personnel_gt50k_n:,}')

# Explicit bin edges so $0 is a boundary; no bin straddles the sign change.
bin_width = 5_000
bin_min = np.floor(_s.min() / bin_width) * bin_width
bin_max = np.ceil(_s.max() / bin_width) * bin_width
bins = np.arange(bin_min, bin_max + bin_width, bin_width)
assert 0.0 in bins, 'Zero must be a bin edge'

counts, bins = np.histogram(_s, bins=bins)

f, ax = plt.subplots(figsize=(8.5, 4.6))
ax.hist(bins[:-1], bins, weights=counts, edgecolor='white', linewidth=0.4)

# Colour bins by sign of the bin (right edge <= 0 is wholly non-positive).
for _i in range(len(bins) - 1):
    ax.patches[_i].set_facecolor(
        PALETTE['negative'] if bins[_i + 1] <= 0 else PALETTE['positive']
    )

ax.set_yscale('log')
ax.set_xlim((-60_000, 100_000))
ax.set_ylim((2e-1, 1e6))
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xlabel('Disbursement amount (USD)')
ax.set_ylabel('Number of disbursements (log scale)')
ax.set_title('Personnel compensation, 112th–117th Congresses')

# Reference markers for the central tendencies the manuscript cites.
ax.axvline(personnel_median, color=PALETTE['neutral'], lw=1.2, ls='--', alpha=0.85)
ax.annotate(
    f'median ${personnel_median:,.0f}', xy=(personnel_median, 5e4),
    xytext=(personnel_median + 8000, 1.5e5),
    fontsize=9, color=PALETTE['neutral'],
    arrowprops=dict(arrowstyle='-', color=PALETTE['neutral'], lw=0.7),
)

# Legend for sign encoding
ax.legend(
    handles=[
        Patch(facecolor=PALETTE['positive'], label='Disbursement (positive)'),
        Patch(facecolor=PALETTE['negative'], label='Reimbursement (negative)'),
    ],
    loc='upper right',
)

f.savefig('personnel_compensation.png')


### Pre- and post-decimal digit distributions (Figure 2)

**Goal.** Test the manuscript claim that values cluster at 'round' numbers — multiples of 10 (the natural left- or right-digit boundary) and multiples of 25 (the anchoring boundary). Deviations from uniform indicate human pricing biases or rounded-number salary anchors.

**Assumptions.** Amounts are taken in absolute value (sign is irrelevant to the digit structure of the magnitude). The two-digit decompositions assume every amount has at least two digits before and exactly two digits after the decimal point.

**Interpretation.** Pre-decimal panel: highest peaks at multiples of 25 (orange) signal anchoring on quarter-hundred values. Post-decimal panel: peaks at both multiples of 10 (red) and multiples of 25 (orange) signal common-cent rounding.

**Validation.** Bar counts sum to the row count of the underlying amount series; the y-axis fractions sum to one within each panel.


In [ ]:
amounts = (
    members_bioguide_df['AMOUNT']
    .dropna()
    .apply(np.abs)
    .map('{:.2f}'.format)
    .astype(str)
)

two_digits_before = amounts.str.split('.').apply(lambda x: str(int(x[0][-2:])))
two_digits_after  = amounts.str.split('.').apply(lambda x: x[1])

two_digits_before_counts = two_digits_before.value_counts()
two_digits_after_counts  = two_digits_after.value_counts()

two_digits_before_counts.index = [
    f'{int(i):02d}' for i in two_digits_before_counts.index
]
two_digits_after_counts.index = [
    f'{int(i):02d}' for i in two_digits_after_counts.index
]

before_frac = two_digits_before_counts / two_digits_before_counts.sum()
after_frac  = two_digits_after_counts  / two_digits_after_counts.sum()


In [ ]:
f, axs = plt.subplots(
    2, 1, figsize=(11, 5.6), sharex=True,
    gridspec_kw={'hspace': 0.35},
)

for ax, frac, label in [
    (axs[0], before_frac, 'Pre-decimal (two digits before the decimal point)'),
    (axs[1], after_frac,  'Post-decimal (two digits after the decimal point)'),
]:
    # Default bar colour
    bars = ax.bar(
        range(len(frac)), frac.sort_index().values,
        color=PALETTE['bar'], width=0.78,
    )
    # Recolour multiples of 25 (orange) and 10 (red, applied second so 50/00 stay red)
    for _i in range(0, 100, 25):
        if _i in range(len(frac)):
            bars[_i].set_color(PALETTE['bar_alt'])
    for _i in range(0, 100, 10):
        if _i in range(len(frac)):
            bars[_i].set_color(PALETTE['bar_warn'])

    ax.set_yscale('log')
    ax.set_ylim((5e-4, 5e-1))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1))
    ax.set_ylabel('Share of disbursements')
    ax.set_title(label, loc='left')
    ax.set_xticks(range(0, 100, 10))
    ax.set_xticklabels(range(0, 100, 10))

axs[1].set_xlabel('Digit value')

f.legend(
    handles=[
        Patch(facecolor=PALETTE['bar'],      label='Other digits'),
        Patch(facecolor=PALETTE['bar_alt'],  label='Multiple of 25'),
        Patch(facecolor=PALETTE['bar_warn'], label='Multiple of 10'),
    ],
    loc='lower center', ncol=3,
    bbox_to_anchor=(0.5, -0.02), frameon=False,
)

f.savefig('pre_post_decimal.png')


### Benford's law across expense categories (Figure 3)

**Goal.** Compare the empirical distribution of leading digits in each expense category against Benford's law, using small multiples so each category can be read without disentangling colour-coded overlays.

**Assumptions.** Benford's law applies to amounts spanning multiple orders of magnitude with no upper cutoff. Personnel-compensation amounts cluster around monthly salaries in a narrow band, so deviation from Benford is expected for that category and is not itself a fraud signal.

**Interpretation.** Each panel shows one expense category's observed leading-digit share (blue bars) against the Benford expectation (black line). Visual deviation is the qualitative cue; the chi-squared statistic in `dist_test_df` (next cell) quantifies it.

**Validation.** The next cell prints the chi-squared and KS test statistics for each category so the visual ordering can be checked against the numerical ranking.


In [ ]:
# Benford expectation
benfords_s = pd.Series({
    str(i): np.log10(1 + 1 / i) for i in range(1, 10)
})

def leading_digits_count(s: pd.Series) -> pd.Series:
    """Return the counts of leading digits 1-9 in a positive-amount Series."""
    _gt0 = s[s > 0]
    _str = _gt0.astype(str)
    _leading = _str.str.get(0)
    _not0 = _leading[_leading != '0']
    return _not0.value_counts().sort_index()


In [ ]:
leading_digit_dist_by_cat = {}
for _cat in members_df['CATEGORY'].value_counts().iloc[:-3].index:
    _amount = members_df.loc[members_df['CATEGORY'] == _cat, 'AMOUNT']
    leading_digit_dist_by_cat[_cat] = leading_digits_count(_amount)

leading_digit_dist_by_cat_df = pd.DataFrame(leading_digit_dist_by_cat)
leading_digit_dist_by_cat_norm_df = (
    leading_digit_dist_by_cat_df / leading_digit_dist_by_cat_df.sum()
)


In [ ]:
# Per-category goodness-of-fit against Benford
dist_test_by_col = {}
for _col in leading_digit_dist_by_cat_df.columns:
    _col_vals = leading_digit_dist_by_cat_df.loc[:, _col]
    _exp_vals = _col_vals.sum() * benfords_s
    _ks   = stats.ks_2samp(_col_vals, _exp_vals)
    _chi2 = stats.chisquare(_col_vals, _exp_vals)
    dist_test_by_col[_col] = {
        'ks_statistic': _ks.statistic,   'ks_pvalue':   _ks.pvalue,
        'chi2_statistic': _chi2.statistic, 'chi2_pvalue': _chi2.pvalue,
    }

dist_test_df = pd.DataFrame(dist_test_by_col).T
dist_test_df.sort_values('chi2_statistic', ascending=False)


In [ ]:
categories = [c for c in top_cats if c in leading_digit_dist_by_cat_norm_df.columns]
n_cats = len(categories)
n_cols = 4
n_rows = int(np.ceil(n_cats / n_cols))

f, axs = plt.subplots(
    n_rows, n_cols,
    figsize=(11.5, 3.2 * n_rows),
    sharex=True, sharey=True,
)
axs = np.array(axs).flatten()

for i, _cat in enumerate(categories):
    ax = axs[i]
    ax.bar(
        range(9), leading_digit_dist_by_cat_norm_df[_cat].values,
        color=PALETTE['bar'], width=0.78, alpha=0.92,
    )
    ax.plot(
        range(9), benfords_s.values,
        color=PALETTE['benford'], lw=2.0, marker='o', markersize=4,
        label="Benford's law", zorder=10,
    )
    ax.set_title(_cat.title(), fontsize=10)
    ax.set_xticks(range(9))
    ax.set_xticklabels(range(1, 10))
    ax.set_ylim((0, 0.45))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    if i % n_cols == 0:
        ax.set_ylabel('Share')
    if i >= n_cats - n_cols:
        ax.set_xlabel('Leading digit')

for j in range(n_cats, len(axs)):
    axs[j].axis('off')

handles, labels = axs[0].get_legend_handles_labels()
f.legend(handles, labels, loc='lower right',
         bbox_to_anchor=(0.99, 0.02), fontsize=10)
f.suptitle("Leading-digit distribution by expense category", fontsize=12, y=1.005)
f.savefig('benford_all_cat.png')


### Member-level Benford deviations — travel (Figure 4)

**Goal.** Identify which members' travel disbursements deviate most and least from Benford's expectation. The small-multiples treatment shows each member's leading-digit profile alongside the Benford line and the across-member median, making the manuscript's qualitative claims about Representative Velázquez (largest deviation) directly inspectable.

**Assumptions.** Restricted to members with more than 1,000 transactions across all categories to avoid noisy small-sample estimates. Member-level Benford analysis uses *travel* amounts only.

**Interpretation.** Top grid: ten members with the largest chi-squared deviation from Benford. Bottom grid: ten with the smallest. The dashed line is the median leading-digit share across all members in the comparison set; the solid line is the Benford expectation.

**Validation.** Test statistics are stored in `dist_test_by_member_travel_df` and ranked by chi-squared statistic — the ranking is what defines the top/bottom 10.


In [ ]:
transaction_count = members_df['BIOGUIDE_ID'].value_counts()
members_gt1000_transactions = transaction_count[transaction_count > 1000].index
print(f'{len(members_gt1000_transactions):,} members with >1,000 disbursements')


In [ ]:
# Per-member leading-digit distribution for travel
leading_digit_dist_by_member_travel = {}
for _member in members_gt1000_transactions:
    _amount = members_df.loc[
        (members_df['BIOGUIDE_ID'] == _member)
        & (members_df['CATEGORY'] == 'TRAVEL'),
        'AMOUNT',
    ]
    leading_digit_dist_by_member_travel[_member] = leading_digits_count(_amount)

leading_digit_dist_by_member_travel_df = pd.DataFrame(
    leading_digit_dist_by_member_travel
)


In [ ]:
# Goodness-of-fit per member (travel)
ks_test_by_member_travel = {}
for _member in leading_digit_dist_by_member_travel_df.columns:
    _col_vals = leading_digit_dist_by_member_travel_df.loc[:, _member]
    _exp_vals = _col_vals.sum() * benfords_s
    _ks   = stats.ks_2samp(_col_vals, _exp_vals)
    _chi2 = stats.chisquare(_col_vals, _exp_vals)
    ks_test_by_member_travel[_member] = {
        'ks_statistic': _ks.statistic, 'ks_pvalue': _ks.pvalue,
        'chi2_statistic': _chi2.statistic, 'chi2_pvalue': _chi2.pvalue,
    }

dist_test_by_member_travel_df = pd.DataFrame(ks_test_by_member_travel).T
dist_test_by_member_travel_df.sort_values('chi2_statistic', ascending=False).head(20)


In [ ]:
leading_digit_dist_by_member_travel_norm_df = (
    leading_digit_dist_by_member_travel_df / leading_digit_dist_by_member_travel_df.sum()
)
top10_members_travel = leading_digit_dist_by_member_travel_norm_df.loc[
    :, dist_test_by_member_travel_df.sort_values('chi2_statistic', ascending=False).head(10).index
]
bot10_members_travel = leading_digit_dist_by_member_travel_norm_df.loc[
    :, dist_test_by_member_travel_df.sort_values('chi2_statistic').head(10).index
]

leading_digit_travel_median = leading_digit_dist_by_member_travel_df.median(1)
leading_digit_travel_median_norm = (
    leading_digit_travel_median / leading_digit_travel_median.sum()
)


Helper for the small-multiples Benford rendering — used twice (travel top/bottom 10; personnel top/bottom 10).

In [ ]:
def plot_member_benford_smallmult(members_to_plot, benfords_s, median_norm,
                                   title, fname, ylim=0.5, n_cols=5):
    """Small-multiples Benford comparison: one panel per member, shared axes.

    Colour convention: blue bars = observed leading-digit share; black line
    with markers = Benford expectation; dashed grey line = median across
    members in the comparison set.
    """
    n = members_to_plot.shape[1]
    n_rows = int(np.ceil(n / n_cols))

    f, axs = plt.subplots(
        n_rows, n_cols, figsize=(14, 2.7 * n_rows),
        sharex=True, sharey=True,
    )
    axs = np.array(axs).flatten()

    for i, member in enumerate(members_to_plot.columns):
        ax = axs[i]
        ax.bar(
            range(9), members_to_plot[member].values,
            color=PALETTE['bar'], width=0.78, alpha=0.92,
        )
        ax.plot(range(9), benfords_s.values,
                color=PALETTE['benford'], lw=1.6, marker='o',
                markersize=3.5, label="Benford")
        ax.plot(range(9), median_norm.values,
                color=PALETTE['neutral'], lw=1.6, ls='--',
                label='Median')
        ax.set_title(member, fontsize=10)
        ax.set_xticks(range(9))
        ax.set_xticklabels(range(1, 10))
        ax.set_ylim((0, ylim))
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
        if i % n_cols == 0:
            ax.set_ylabel('Share')
        if i >= n - n_cols:
            ax.set_xlabel('Leading digit')

    for j in range(n, len(axs)):
        axs[j].axis('off')

    handles, labels = axs[0].get_legend_handles_labels()
    f.legend(handles[-2:], labels[-2:],
             loc='lower right', bbox_to_anchor=(0.99, 0.02),
             fontsize=10, ncol=2)
    f.suptitle(title, fontsize=12, y=1.005)
    f.savefig(fname)
    return f


In [ ]:
plot_member_benford_smallmult(
    top10_members_travel, benfords_s, leading_digit_travel_median_norm,
    "Largest deviations from Benford's law — travel expenditures",
    'benford_travel_top10.png',
)


In [ ]:
plot_member_benford_smallmult(
    bot10_members_travel, benfords_s, leading_digit_travel_median_norm,
    "Smallest deviations from Benford's law — travel expenditures",
    'benford_travel_bot10.png',
)


### Member-level Benford deviations — personnel (supplementary)

**Goal.** Same small-multiples treatment for personnel-compensation amounts. The manuscript discusses these only briefly; the panels are kept in the notebook for completeness and as supplementary material.

**Assumptions.** Same 1,000-transaction restriction. Personnel amounts cluster near monthly-salary anchor values, so deviation from Benford is the baseline expectation rather than a flag.

**Interpretation.** Even the 'smallest deviation' panel for personnel deviates from Benford because salary anchors dominate the leading-digit distribution at $9,000 and $10,000.

**Validation.** Tests stored in `dist_test_by_member_personnel_df`; ranking drives the top/bottom selection.


In [ ]:
leading_digit_dist_by_member_personnel = {}
for _member in members_gt1000_transactions:
    _amount = members_df.loc[
        (members_df['BIOGUIDE_ID'] == _member)
        & (members_df['CATEGORY'] == 'PERSONNEL COMPENSATION'),
        'AMOUNT',
    ]
    leading_digit_dist_by_member_personnel[_member] = leading_digits_count(_amount)

leading_digit_dist_by_member_personnel_df = pd.DataFrame(
    leading_digit_dist_by_member_personnel
)


In [ ]:
ks_test_by_member_personnel = {}
for _member in leading_digit_dist_by_member_personnel_df.columns:
    _col_vals = leading_digit_dist_by_member_personnel_df.loc[:, _member]
    _exp_vals = _col_vals.sum() * benfords_s
    _ks   = stats.ks_2samp(_col_vals, _exp_vals)
    _chi2 = stats.chisquare(_col_vals, _exp_vals)
    ks_test_by_member_personnel[_member] = {
        'ks_statistic': _ks.statistic, 'ks_pvalue': _ks.pvalue,
        'chi2_statistic': _chi2.statistic, 'chi2_pvalue': _chi2.pvalue,
    }

dist_test_by_member_personnel_df = pd.DataFrame(ks_test_by_member_personnel).T

leading_digit_dist_by_member_personnel_norm_df = (
    leading_digit_dist_by_member_personnel_df / leading_digit_dist_by_member_personnel_df.sum()
)
top10_members_personnel = leading_digit_dist_by_member_personnel_norm_df.loc[
    :, dist_test_by_member_personnel_df.sort_values('chi2_statistic', ascending=False).head(10).index
]
bot10_members_personnel = leading_digit_dist_by_member_personnel_norm_df.loc[
    :, dist_test_by_member_personnel_df.sort_values('chi2_statistic').head(10).index
]

leading_digit_personnel_median = leading_digit_dist_by_member_personnel_df.median(1)
leading_digit_personnel_median_norm = (
    leading_digit_personnel_median / leading_digit_personnel_median.sum()
)


In [ ]:
plot_member_benford_smallmult(
    top10_members_personnel, benfords_s, leading_digit_personnel_median_norm,
    "Largest deviations from Benford's law — personnel",
    'benford_personnel_top10.png', ylim=0.6,
)


In [ ]:
plot_member_benford_smallmult(
    bot10_members_personnel, benfords_s, leading_digit_personnel_median_norm,
    "Smallest deviations from Benford's law — personnel",
    'benford_personnel_bot10.png', ylim=0.6,
)


### Relative size factor (Figure 5)

**Goal.** Show the distribution of within-member, within-year, within-category ratios between the largest and second-largest expense. Members with anomalously bundled spending (one large entry plus many small ones) sit far in the tail.

**Assumptions.** Both the largest and second-largest expense in each group must be positive — negative reimbursements would invert the ratio's meaning.

**Interpretation.** The bulk of ratios live between 1 and 10. Ratios above 500 are concentrated in `FRANKED MAIL` for offices that bundle a quarter's franking into a single line item alongside small miscellaneous charges (Representative Payne's pattern).

**Validation.** Pinned headline figures (share of ratios ≤ 10, count of ratios > 500) are printed below.


In [ ]:
# Within (member, year, category): ratio of largest to second-largest expense
two_largest_expenses = (
    members_df.groupby(['BIOGUIDE_ID', 'YEAR', 'CATEGORY'])
    .agg({'AMOUNT': lambda x: x.nlargest(2)})['AMOUNT']
)

# Keep only groups with at least two entries, both positive
two_largest_expenses = two_largest_expenses[
    two_largest_expenses.apply(lambda x: isinstance(x, np.ndarray))
]
two_largest_expenses = two_largest_expenses[
    two_largest_expenses.apply(lambda x: x[0] > 0)
    & two_largest_expenses.apply(lambda x: x[1] > 0)
]

largest_expenses_ratio = two_largest_expenses.apply(lambda x: x[0] / x[1])

rsf_le10_pct  = (largest_expenses_ratio <= 10).mean()
rsf_gt500_n   = int((largest_expenses_ratio > 500).sum())
rsf_max       = largest_expenses_ratio.max()
print(f'N (member-year-category) groups:   {len(largest_expenses_ratio):,}')
print(f'Share of ratios <= 10:             {rsf_le10_pct:.1%}')
print(f'Count of ratios > 500:             {rsf_gt500_n}')
print(f'Max ratio:                         {rsf_max:,.0f}')


In [ ]:
f, ax = plt.subplots(figsize=(8.5, 4.5))

bins = np.logspace(0, 4, 25)
ax.hist(largest_expenses_ratio, bins=bins,
        color=PALETTE['bar'], edgecolor='white', linewidth=0.4)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim((1, 1e4))
ax.set_ylim((1, 1e5))
ax.set_xlabel('Ratio of largest to second-largest expense (log scale)')
ax.set_ylabel('Count of member-year-category groups (log scale)')
ax.set_title('Relative size factor across member-year-category groups')

# Reference markers
ax.axvline(10, color=PALETTE['neutral'], lw=1, ls='--')
ax.axvline(500, color=PALETTE['highlight'], lw=1, ls='--')
ax.annotate(f'{rsf_le10_pct:.0%} ≤ 10', xy=(10, 4e4),
            xytext=(2.5, 4.5e4), fontsize=9, color=PALETTE['neutral'])
ax.annotate(f'{rsf_gt500_n} > 500', xy=(500, 4e2),
            xytext=(1200, 6e2), fontsize=9, color=PALETTE['highlight'])

f.savefig('two_largest_ratio.png')


## Categorical anomalies

*Maps to manuscript §Exploratory Data Auditing → Categorical anomalies.*

Three analyses: the quarterly categorical-expense profile (Figure 6) and the by-gender (Figure 7) and by-party (Figure 8) comparisons of category-level spending.

### Quarterly categorical expense profiles (Figure 6)

**Goal.** Identify member-quarters whose categorical expense-count distribution departs most sharply from the median profile across all member-quarters. The four highest-ranked anomalies are the manuscript's worked examples.

**Assumptions.** Member-quarters with fewer than 20 total expenses across the eight categories are excluded — small denominators produce unreliable chi-squared statistics. `PERSONNEL BENEFITS`, `TRANSPORTATION OF THINGS`, and `BENEFITS TO FORMER PERSONNEL` are dropped because they are sparse and not in `top_cats`. The radar plot uses *share of quarterly expenses* per category — the same probability scale the chi-squared statistic compares — so every axis runs 0 to ~50% on a directly comparable scale.

**Interpretation.** Each axis is one expense category; the polygon's distance from the centre on that axis is the category's share of the member-quarter's total expenses. The black polygon shows the population median profile and the blue band shows the population interquartile range (25th–75th percentile); together they describe what a *typical* member-quarter's categorical mix looks like. The four coloured polygons are the four member-quarter profiles with the smallest chi-squared p-values relative to the median profile — visual deviation from the black reference is the signal of anomalousness, and the angular position of the deviation tells you which category is driving it. Category axes are ordered clockwise by median share so the largest categories appear together.

**Validation.** Random sample seed and chi-squared computation are unchanged from the previous (profile-plot) rendering; only the visual encoding changes. The radial-axis upper limit is set to fit the highest peak across all five overlaid polygons so no profile is visually clipped.

In [ ]:
member_expense_count_by_category_df = (
    members_df.groupby(['YEAR-QUARTER', 'BIOGUIDE_ID'])
    ['CATEGORY']
    .value_counts()
    .unstack(-1)
    .fillna(0)
)
member_expense_count_by_category_df.drop(
    columns=['BENEFITS TO FORMER PERSONNEL',
             'PERSONNEL BENEFITS', 'TRANSPORTATION OF THINGS'],
    inplace=True,
)

# Restrict to member-quarters with at least 100 expenses
gt20 = member_expense_count_by_category_df.sum(1)
gt20 = gt20[gt20 >= 100]
gt20 = member_expense_count_by_category_df.loc[gt20.index]

# Population-median categorical profile (as a probability)
_exp = gt20.median(axis=0)
_exp = _exp / _exp.sum()
_df  = gt20.div(gt20.sum(1), axis=0)

chisquare_pvalues_expense_counts = pd.Series(
    data=stats.chisquare(_df, _exp, axis=1).pvalue,
    index=gt20.index,
)
chisquare_pvalues_expense_counts.sort_values().head(10)


In [ ]:
gt20.head()

In [ ]:
# Radar (polar) plot of categorical expense profiles. Each axis is one
# expense category; radial position is the share of the member-quarter's
# total expenses falling in that category. The chi-squared test uses the
# same share scale, so the visual deviation between a coloured anomaly
# polygon and the black median polygon corresponds directly to what the
# inferential statistic is detecting.

_order = member_expense_count_by_category_df.median().sort_values().index.tolist()
top4_anomalies = chisquare_pvalues_expense_counts.sort_values().head(4)

# Population statistics on the share scale (matches the chi-squared formulation).
shares_df = member_expense_count_by_category_df.div(
    member_expense_count_by_category_df.sum(1), axis=0
)
median_shares = shares_df[_order].median()
q25_shares    = shares_df[_order].quantile(0.25)
q75_shares    = shares_df[_order].quantile(0.75)

# Shortened category labels so 8 axes stay legible at standard print size.
CAT_SHORT = {
    'PERSONNEL COMPENSATION':       'Personnel',
    'FRANKED MAIL':                 'Franked\nMail',
    'TRAVEL':                       'Travel',
    'RENT COMMUNICATION UTILITIES': 'Rent / Comm /\nUtils',
    'PRINTING AND REPRODUCTION':    'Printing &\nReproduction',
    'OTHER SERVICES':               'Other\nServices',
    'SUPPLIES AND MATERIALS':       'Supplies &\nMaterials',
    'EQUIPMENT':                    'Equipment',
}

# Radar geometry: one angle per category, repeated at the end to close the polygon.
n_cats = len(_order)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

def _close(seq):
    """Repeat the first element at the end so a polygon closes cleanly."""
    return list(seq) + [seq[0]]

f, ax = plt.subplots(figsize=(10.5, 9.0), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)   # category 0 at top
ax.set_theta_direction(-1)       # clockwise (standard radar convention)

# Population IQR — annular band between the 25th and 75th percentile polygons
ax.fill_between(
    angles, _close(q25_shares.values), _close(q75_shares.values),
    color=PALETTE['band'], alpha=0.55,
    label='Population IQR (25th\u201375th pct)',
)

# Population median — the reference polygon
ax.plot(
    angles, _close(median_shares.values),
    color=PALETTE['benford'], lw=2.6, zorder=5,
    label='Population median',
)

# Four anomalous member-quarters, each as a coloured polygon overlay
ANOMALY_COLORS = ['#A6332E', '#D67D2C', '#7B5BA6', '#3B7A8A']
anomaly_peaks = []
for i, ((yq, bio_id), pval) in enumerate(top4_anomalies.items()):
    row = shares_df.loc[(yq, bio_id), _order]
    anomaly_peaks.append(row.max())
    ax.plot(
        angles, _close(row.values),
        color=ANOMALY_COLORS[i], lw=2.0, alpha=0.92,
        marker='o', markersize=5,
        markeredgecolor='white', markeredgewidth=0.6,
        label=f'{bio_id}, {yq}  (\u03c7\u00b2 p = {pval:.1e})',
    )

# Axis cosmetics
ax.set_xticks(angles[:-1])
# ax.set_xscale('log')
ax.set_xticklabels(
    [CAT_SHORT.get(c, c.title()) for c in _order],
    fontsize=9,
)

# Radial range fits the highest peak across all overlaid polygons,
# with a small headroom so the outermost vertex isn't on the axis edge.
r_max = max(q75_shares.max(), max(anomaly_peaks)) * 1.10
ax.set_ylim(0, r_max)
ax.set_rlabel_position(22.5)  # radial-tick labels between the first two axes
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.tick_params(axis='y', labelsize=8, colors=PALETTE['neutral'])
ax.grid(True, color='#dddddd', linewidth=0.6)
ax.spines['polar'].set_color('#bbbbbb')
ax.spines['polar'].set_linewidth(0.8)

ax.legend(
    loc='upper left', bbox_to_anchor=(1.10, 1.0),
    fontsize=9, frameon=False, borderaxespad=0,
)
ax.set_title(
    'Categorical expense profiles: typical and anomalous',
    fontsize=12, y=1.10,
)

f.savefig('categorical_anomalies_profiles.png')


Build the cleaned member-quarter-category sums used by Figures 7 and 8.

In [ ]:
member_quarterly_category_amount_df = (
    members_df.groupby(['YEAR-QUARTER', 'BIOGUIDE_ID', 'CATEGORY'])
    .agg({'AMOUNT': 'sum', 'OFFICE': len})
    .reset_index()
    .rename(columns={'OFFICE': 'EXPENSE COUNT'})
)

member_quarterly_category_amount_df = pd.merge(
    left=member_quarterly_category_amount_df,
    right=propublica_members_df[['id', 'gender', 'party']].drop_duplicates(subset=['id']),
    left_on='BIOGUIDE_ID', right_on='id', how='left',
)

# Restrict to (a) member-quarter-category rows with at least 5 expenses,
# (b) the eight informative categories, (c) members with party in {D, R}.
c0 = member_quarterly_category_amount_df['EXPENSE COUNT'] >= 5
c1 = ~member_quarterly_category_amount_df['CATEGORY'].isin(
    ['PERSONNEL BENEFITS', 'TRANSPORTATION OF THINGS'])
c2 = member_quarterly_category_amount_df['party'].isin(['D', 'R'])

clean_bio_cat_df = member_quarterly_category_amount_df[c0 & c1 & c2]
print(f'{len(clean_bio_cat_df):,} member-quarter-category observations after filters')


### Quarterly spending by gender (Figure 7)

**Goal.** Compare category-level quarterly spending between male and female members using paired boxplots on a log x-axis, supplemented by both parametric (log-transformed Welch) and nonparametric (Mann–Whitney U) inferential tests.

**Assumptions.** Amounts are heavily right-skewed, so a t-test on the raw scale is untrustworthy; the log10 transformation linearises the inferential scale. The Mann–Whitney U is added as a distribution-free check. Sample sizes per group are reported; effect sizes accompany every p-value.

**Interpretation.** Boxplots show the conditional distribution (median, IQR, whiskers; outliers hidden for clarity). The printed table reports N per group, Welch's t on log10 amounts, Cohen's d on the log scale, and Mann–Whitney's p-value and rank-biserial r.

**Validation.** Sample sizes per category are explicitly checked (≥5 per group required) before any test is run. Rows with zero amount are excluded from the log transform.


In [ ]:
# Sample sizes
gender_counts = clean_bio_cat_df.groupby('gender').size()
gender_unique_members = clean_bio_cat_df.groupby('gender')['BIOGUIDE_ID'].nunique()
print('Sample sizes:')
for g in ['M', 'F']:
    n_obs = int(gender_counts.get(g, 0))
    n_mem = int(gender_unique_members.get(g, 0))
    print(f'  {g}: {n_obs:,} member-quarter-category observations, '
          f'{n_mem:,} unique members')
print()

# Paired boxplots
_order = clean_bio_cat_df.groupby('CATEGORY')['AMOUNT'].median().sort_values().index.tolist()
gender_palette = {'F': PALETTE['F'], 'M': PALETTE['M']}

f, ax = plt.subplots(figsize=(10, 5.8))
sb.boxplot(
    data=clean_bio_cat_df, x='AMOUNT', y='CATEGORY', hue='gender',
    order=_order, palette=gender_palette, showfliers=False,
    width=0.72, linewidth=0.9, ax=ax,
)
ax.set_xscale('log')
ax.set_xlim((1e2, 1e7))
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xlabel('Quarterly category expenditure (USD, log scale)')
ax.set_ylabel(None)
ax.legend(title='Gender', loc='lower right')
ax.set_title('Quarterly category spending by gender', loc='left')

f.savefig('category_spending_gender.png')


In [ ]:
# Per-category statistical tests — log10 amounts
print('Per-category tests (log10 amounts):')
print(f"{'Category':<32s} {'N_M':>5s} {'N_F':>5s} "
      f"{'t(log)':>7s} {'p(t)':>9s} {'d':>6s} "
      f"{'p(MWU)':>9s} {'rb':>6s}")

for _cat in _order:
    _c0 = clean_bio_cat_df['CATEGORY'] == _cat
    _men = clean_bio_cat_df.loc[_c0 & (clean_bio_cat_df['gender'] == 'M'), 'AMOUNT']
    _women = clean_bio_cat_df.loc[_c0 & (clean_bio_cat_df['gender'] == 'F'), 'AMOUNT']
    n1, n2 = len(_men), len(_women)
    if n1 < 5 or n2 < 5:
        continue

    _men_log = np.log10(_men[_men > 0])
    _women_log = np.log10(_women[_women > 0])
    t_log = stats.ttest_ind(_men_log, _women_log, equal_var=False)
    pooled_sd = np.sqrt(
        (_men_log.var(ddof=1) + _women_log.var(ddof=1)) / 2
    )
    cohens_d = (
        (_men_log.mean() - _women_log.mean()) / pooled_sd
        if pooled_sd > 0 else np.nan
    )
    mw = stats.mannwhitneyu(_men, _women, alternative='two-sided')
    rb = 1 - 2 * mw.statistic / (n1 * n2)

    print(f'{_cat[:32]:<32s} {n1:>5d} {n2:>5d} '
          f'{t_log.statistic:>7.2f} {t_log.pvalue:>9.2e} {cohens_d:>6.2f} '
          f'{mw.pvalue:>9.2e} {rb:>6.2f}')


### Quarterly spending by party (Figure 8)

**Goal.** Same treatment as Figure 7, comparing Democratic and Republican members. Manuscript text claims significant differences in every category — the printed table here lets a critic check the direction and magnitude.

**Assumptions.** Same: amounts heavily right-skewed → tests on log10 amounts; Mann–Whitney as nonparametric check; effect sizes reported. Independent members are excluded; only D and R retained.

**Interpretation.** Boxplots show the within-category distribution side-by-side; the table prints all four statistics per category.

**Validation.** Same N ≥ 5 per group check before testing.


In [ ]:
party_subset = clean_bio_cat_df[clean_bio_cat_df['party'].isin(['D', 'R'])]

party_counts = party_subset.groupby('party').size()
party_unique_members = party_subset.groupby('party')['BIOGUIDE_ID'].nunique()
print('Sample sizes:')
for p in ['D', 'R']:
    n_obs = int(party_counts.get(p, 0))
    n_mem = int(party_unique_members.get(p, 0))
    print(f'  {p}: {n_obs:,} member-quarter-category observations, '
          f'{n_mem:,} unique members')
print()

_order = clean_bio_cat_df.groupby('CATEGORY')['AMOUNT'].median().sort_values().index.tolist()
party_palette = {'D': PALETTE['D'], 'R': PALETTE['R']}

f, ax = plt.subplots(figsize=(10, 5.8))
sb.boxplot(
    data=party_subset, x='AMOUNT', y='CATEGORY', hue='party',
    hue_order=['D', 'R'], order=_order, palette=party_palette,
    showfliers=False, width=0.72, linewidth=0.9, ax=ax,
)
ax.set_xscale('log')
ax.set_xlim((1e2, 1e7))
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xlabel('Quarterly category expenditure (USD, log scale)')
ax.set_ylabel(None)
ax.legend(title='Party', loc='lower right')
ax.set_title('Quarterly category spending by party', loc='left')

f.savefig('category_spending_party.png')


In [ ]:
print('Per-category tests (log10 amounts):')
print(f"{'Category':<32s} {'N_D':>5s} {'N_R':>5s} "
      f"{'t(log)':>7s} {'p(t)':>9s} {'d':>6s} "
      f"{'p(MWU)':>9s} {'rb':>6s}")

for _cat in _order:
    _c0 = party_subset['CATEGORY'] == _cat
    _dem = party_subset.loc[_c0 & (party_subset['party'] == 'D'), 'AMOUNT']
    _rep = party_subset.loc[_c0 & (party_subset['party'] == 'R'), 'AMOUNT']
    n1, n2 = len(_dem), len(_rep)
    if n1 < 5 or n2 < 5:
        continue

    _dem_log = np.log10(_dem[_dem > 0])
    _rep_log = np.log10(_rep[_rep > 0])
    t_log = stats.ttest_ind(_dem_log, _rep_log, equal_var=False)
    pooled_sd = np.sqrt(
        (_dem_log.var(ddof=1) + _rep_log.var(ddof=1)) / 2
    )
    cohens_d = (
        (_dem_log.mean() - _rep_log.mean()) / pooled_sd
        if pooled_sd > 0 else np.nan
    )
    mw = stats.mannwhitneyu(_dem, _rep, alternative='two-sided')
    rb = 1 - 2 * mw.statistic / (n1 * n2)

    print(f'{_cat[:32]:<32s} {n1:>5d} {n2:>5d} '
          f'{t_log.statistic:>7.2f} {t_log.pvalue:>9.2e} {cohens_d:>6.2f} '
          f'{mw.pvalue:>9.2e} {rb:>6.2f}')


## Temporal anomalies

*Maps to manuscript §Exploratory Data Auditing → Temporal anomalies.*

Three analyses: the COVID-19 impact on travel sub-categories (Figure 9), cyclical spending across the eight-quarter congressional term (Figure 10), and the inter-event timing distribution (Figure 11).

### COVID-19 impact on travel sub-categories (Figure 9)

**Goal.** Track how three travel sub-categories — commercial travel (airfare/trains), car travel (mileage/gas/tolls), and lodging-and-meals — moved relative to their 2019 averages across the pandemic disruption and recovery.

**Assumptions.** Disbursements are aggregated by `YEAR-QUARTER` and normalised by each sub-category's 2019 quarterly mean, so the y-axis is a unit-free ratio. The Welch's t comparison between the 12 pandemic quarters (2020Q1–2022Q4) and the 12 preceding quarters (2017Q1–2019Q4) tests whether car-travel spending stayed structurally lower.

**Interpretation.** A pre-pandemic baseline of 1.0; the 2020Q2 trough marker is the start of national shutdowns. Commercial travel and car travel remained depressed for longer than lodging-and-meals, which recovered to and above the 2019 baseline by 2022.

**Validation.** The Welch's t-test result (`covid_car_ttest`) is printed so the manuscript's structural-decline claim for car travel can be checked. The 2019-mean denominator is reproducible from the underlying data.


In [ ]:
members_travel_df = members_df[members_df['CATEGORY'] == 'TRAVEL']

_df0 = members_travel_df[members_travel_df['PURPOSE'].isin(
    ['COMMERCIAL TRANSPORTATION', 'AIRFARE COMMERCIAL TRANSPORT'])]
_df1 = members_travel_df[members_travel_df['PURPOSE'].isin(
    ['PRIVATE AUTO MILEAGE', 'GASOLINE', 'TAXI/PARKING/TOLLS'])]
_df2 = members_travel_df[members_travel_df['PURPOSE'].isin(
    ['LODGING', 'MEALS'])]

_df0_agg = _df0.groupby('YEAR-QUARTER').agg({'AMOUNT': 'sum'})['AMOUNT']
_df1_agg = _df1.groupby('YEAR-QUARTER').agg({'AMOUNT': 'sum'})['AMOUNT']
_df2_agg = _df2.groupby('YEAR-QUARTER').agg({'AMOUNT': 'sum'})['AMOUNT']

_2019 = ['2019Q1', '2019Q2', '2019Q3', '2019Q4']
_df0_norm = _df0_agg / _df0_agg.loc[_2019].mean()
_df1_norm = _df1_agg / _df1_agg.loc[_2019].mean()
_df2_norm = _df2_agg / _df2_agg.loc[_2019].mean()

# Welch's t on car-travel sums: pandemic (24:36 vs 36:48 of the time index)
covid_car_ttest = stats.ttest_ind(
    _df1_agg.iloc[36:48], _df1_agg.iloc[24:36], equal_var=False
)
print(f'Car-travel: pandemic (12q) vs prior (12q): '
      f't = {covid_car_ttest.statistic:.2f}, p = {covid_car_ttest.pvalue:.2e}')


In [ ]:
f, ax = plt.subplots(figsize=(9.5, 5.0))

for series, label, color in [
    (_df0_norm, 'Commercial travel', PALETTE['commercial']),
    (_df1_norm, 'Car travel',        PALETTE['car']),
    (_df2_norm, 'Lodging & meals',   PALETTE['lodging']),
]:
    ax.plot(series.index, series.values, lw=2.6, label=label, color=color,
            marker='o', markersize=3, markeredgecolor='white', markeredgewidth=0.6)

ax.axhline(1.0, color=PALETTE['neutral'], lw=0.8, ls=':')
ax.axvline('2020Q2', color=PALETTE['highlight'], lw=1, ls='--', alpha=0.9)
ax.annotate(
    'COVID-19 shutdowns', xy=('2020Q2', 1.4), xytext=('2017Q2', 1.65),
    color=PALETTE['highlight'], fontsize=10,
    arrowprops=dict(arrowstyle='-', color=PALETTE['highlight'], lw=0.8),
)

ax.set_ylim((0, 2))
ax.set_ylabel('Spending (normalized to 2019 mean = 1.0)')
ax.set_xlabel('Quarter')
ax.set_title('Travel sub-categories across the COVID-19 disruption', loc='left')
ax.legend(loc='upper right')

# Rotate x-tick labels and show every fourth tick (annual)
_labels = list(_df0_norm.index)
ax.set_xticks(range(0, len(_labels), 4))
ax.set_xticklabels([_labels[i] for i in range(0, len(_labels), 4)],
                   rotation=45, ha='right')

f.savefig('temporal_covid_shutdowns.png')


### Cyclical spending across the eight-quarter term (Figure 10)

**Goal.** Show how spending varies across the eight quarters of a member's two-year term, by category. Each panel is one category; the box plot displays distribution shape rather than just central tendency.

**Assumptions.** `TERM_QUARTER` is computed by the cleaning notebook as 1–8 across the term. Only positive amounts are kept for the log scale. The three sparse categories (`PERSONNEL BENEFITS`, etc.) are dropped.

**Interpretation.** Categories like `PERSONNEL COMPENSATION` and `RENT COMMUNICATION UTILITIES` are stable across the term — long-running contracts. `EQUIPMENT`, `SUPPLIES AND MATERIALS`, and `PRINTING AND REPRODUCTION` show pronounced cyclicality (set-up in Q1, end-of-term spend-down).

**Validation.** Each panel uses the same x-axis (term quarter 1–8) and log-y so panels can be cross-compared. Outliers are hidden by `showfliers=False` to keep the distribution shape visible.


In [ ]:
populated_categories = members_df['CATEGORY'].value_counts().index[:-3]
member_termly_spending_by_category_df = members_df[
    members_df['CATEGORY'].isin(populated_categories)
]
member_termly_spending_by_category_df = (
    member_termly_spending_by_category_df
    .groupby(['BIOGUIDE_ID', 'YEAR', 'TERM_QUARTER', 'CATEGORY'])
    .agg({'AMOUNT': 'sum'})
    .reset_index()
)


In [ ]:
cats_to_plot = sorted(
    member_termly_spending_by_category_df['CATEGORY'].unique()
)
n_cats = len(cats_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_cats / n_cols))

f, axs = plt.subplots(n_rows, n_cols, figsize=(14, 3.8 * n_rows), sharex=True)
axs = np.array(axs).flatten()

for i, _cat in enumerate(cats_to_plot):
    ax = axs[i]
    _sub_pos = member_termly_spending_by_category_df[
        (member_termly_spending_by_category_df['CATEGORY'] == _cat)
        & (member_termly_spending_by_category_df['AMOUNT'] > 0)
    ]
    sb.boxplot(
        data=_sub_pos, x='TERM_QUARTER', y='AMOUNT',
        ax=ax, color=PALETTE['bar'],
        showfliers=False, width=0.62, linewidth=0.8,
    )
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
    ax.set_title(_cat.title(), fontsize=10)
    ax.set_xlabel('Term quarter (1–8)' if i >= n_cats - n_cols else '')
    ax.set_ylabel('Quarterly spend' if i % n_cols == 0 else '')

for j in range(n_cats, len(axs)):
    axs[j].axis('off')

f.suptitle('Office spending by term quarter, across categories',
           fontsize=12, y=1.005)
f.savefig('spending_term_quarter.png')


### Inter-event timing for expenses (Figure 11)

**Goal.** Characterise the distribution of time between successive expenses within each member-category. Periodic structures (biweekly payroll, monthly billing, quarterly reconciliation, two-year term boundaries) should appear as bumps in the empirical distribution.

**Assumptions.** The 'inter-event time' is computed within each (`BIOGUIDE_ID`, `CATEGORY`) group on dates that are already datetime-parsed by the cleaning notebook. Same-day events (zero lag) are excluded from the histogram bins shown below — they account for ~56% of pairs and would dominate the visualisation.

**Interpretation.** Two panels: a linear-binned histogram emphasising the meaningful 0–800-day range (left), and a complementary CDF on log–log axes that makes the long-lag tail interpretable (right). Vertical markers identify expected cycles (2 weeks, 1 month, 1 quarter, 2 years).

**Validation.** The CCDF is computed from sorted lags and matches the histogram where they overlap. Same-day-pair share is printed for reference.


In [ ]:
interevent_data = (
    members_df.copy()
    .sort_values(['BIOGUIDE_ID', 'CATEGORY', 'DATE'], ascending=True)
    .groupby(['BIOGUIDE_ID', 'CATEGORY'])['DATE']
    .diff() / pd.Timedelta(1, 'd')
)
interevent_data = interevent_data.dropna()

same_day_share = (interevent_data == 0).mean()
print(f'N inter-event pairs:      {len(interevent_data):,}')
print(f'Same-day pair share:      {same_day_share:.1%}')


In [ ]:
f, axs = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Panel A: linear bins
ax = axs[0]
bins = np.arange(0, interevent_data.max() + 7, 7)
ax.hist(interevent_data, bins=bins,
        color=PALETTE['bar'], edgecolor='white', linewidth=0.3)
ax.set_yscale('log')
ax.set_xlim((0, 800))
ax.set_ylim((1, 1e7))
ax.set_xlabel('Days between expenses')
ax.set_ylabel('Count (log scale)')
ax.set_title('Linear bins (7-day width)', loc='left')

for label, x, color in CYCLES:
    if x <= ax.get_xlim()[1]:
        ax.axvline(x, color=color, lw=1, ls='--', label=label, alpha=0.9)
ax.legend(fontsize=9, loc='upper right')

# Panel B: complementary CDF
ax = axs[1]
sorted_lags = np.sort(interevent_data.values)
ccdf = 1.0 - np.arange(1, len(sorted_lags) + 1) / len(sorted_lags)
ax.plot(sorted_lags, ccdf, color=PALETTE['bar'], lw=1.8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim((1, 1e4))
ax.set_xlabel('Days between expenses (log scale)')
ax.set_ylabel('P(lag > x)')
ax.set_title('Complementary CDF', loc='left')

for label, x, color in CYCLES:
    ax.axvline(x, color=color, lw=1, ls='--', label=label, alpha=0.9)
ax.legend(fontsize=9, loc='lower left')

f.suptitle('Inter-event timing between successive expenses',
           fontsize=12, y=1.005)
f.savefig('interevent_anomalies.png')


## Textual anomalies

*Maps to manuscript §Exploratory Data Auditing → Textual anomalies.*

No figure in the manuscript. The textual-anomaly section documents formatting drift in the `PAYEE` column and the cleanup procedure that normalises names across the format breaks at 2016Q4 (commas drop) and 2019Q1 (given-name leading style introduced). The personnel `PAYEE_CLEANED` column built here is consumed by the relational-anomaly analysis below.

### Counts of PAYEE formats per quarter

**Goal.** Document the three name-formatting regimes the manuscript references and verify the two transition quarters (2016Q4, 2019Q1).

**Assumptions.** The presence of a comma in any `PAYEE` cell indicates `{Surname}, {Given Name}` style. The absence of commas and the presence of multi-token names indicates the later styles.

**Interpretation.** Quarters with non-zero comma counts use the early `{Surname}, {Given Name}` style. The cleanup function below harmonises names across all three styles into a consistent `{Given Name} ... {Surname}` form.

**Validation.** The two transition quarters appear at the expected positions in the printed series.


In [ ]:
personnel_compensation_df = members_df[
    members_df['CATEGORY'] == 'PERSONNEL COMPENSATION'
]

yq_total_personnel = personnel_compensation_df.groupby(
    ['YEAR-QUARTER', 'BIOGUIDE_ID', 'PAYEE']
).agg({'AMOUNT': 'sum'}).reset_index()
yq_total_personnel['PAYEE_CLEANED'] = pd.NA

_s = yq_total_personnel.groupby('YEAR-QUARTER').agg(
    {'PAYEE': lambda x: x.str.contains(',').sum()}
)['PAYEE']
comma_in_name_quarters = _s[_s > 0].index
print('Quarters with comma-formatted names:',
      list(comma_in_name_quarters)[:3], '...',
      list(comma_in_name_quarters)[-3:])


### PAYEE name normalisation

**Goal.** Convert every PAYEE string into a canonical `{Given Name} ... {Surname}` form so that staff appearing in multiple formats across the three regimes are recognised as the same person by the relational-anomaly analysis.

**Assumptions.** Names have between 2 and 5 space-separated tokens. The `name_reorganizer` function applies the post-2016Q4 reordering rules for two-, three-, four-, and five-token names. One known alias is patched manually (`T E ANFINSON` → `THOMAS E ANFINSON`).

**Interpretation.** `PAYEE_CLEANED` is the column the relational analysis uses; it carries a uniform `Given ... Surname` form across all quarters.

**Validation.** Spot-check: `yq_total_personnel.loc[..., 'PAYEE_CLEANED']` should contain no commas after normalisation.


In [ ]:
def name_reorganizer(s):
    """Reorder a space-separated name from `{Surname Given ...}` into
    `{Given ... Surname}` form, dispatching on token count.
    """
    _split = s.strip().split(' ')
    n = len(_split)
    if n == 1:
        return _split[0]
    if n == 2:
        return _split[-1] + ' ' + _split[0]
    if n == 3:
        return _split[-2] + ' ' + _split[-1] + ' ' + _split[0]
    if n == 4:
        return _split[-2] + ' ' + _split[-1] + ' ' + _split[0] + ' ' + _split[1]
    if n == 5:
        return (_split[-2] + ' ' + _split[-1] + ' '
                + _split[0] + ' ' + _split[1] + ' ' + _split[2])
    return s


In [ ]:
# Pre-2016Q4: comma-separated `Surname, Given Name` → re-split and reorder
pre_2016Q4 = yq_total_personnel['YEAR-QUARTER'].isin(comma_in_name_quarters)
name_array_len_sep_comma = (
    yq_total_personnel.loc[pre_2016Q4, 'PAYEE'].str.split(',').apply(len)
)
exactly_2 = name_array_len_sep_comma[name_array_len_sep_comma == 2]
pre_2016Q4_cleaned_names = (
    yq_total_personnel.loc[exactly_2.index, 'PAYEE']
    .str.split(',').apply(lambda x: x[1] + ' ' + x[0])
    .str.replace('  ', ' ').str.strip()
)

# Post-2016Q4: space-separated; reorder by token count
post_2016Q4 = ~yq_total_personnel['YEAR-QUARTER'].isin(comma_in_name_quarters)
post_2016Q4_cleaned_names = (
    yq_total_personnel.loc[post_2016Q4, 'PAYEE']
    .str.replace('  ', ' ').str.strip()
    .apply(name_reorganizer)
)

yq_total_personnel.loc[pre_2016Q4_cleaned_names.index, 'PAYEE_CLEANED'] = \
    pre_2016Q4_cleaned_names
yq_total_personnel.loc[post_2016Q4_cleaned_names.index, 'PAYEE_CLEANED'] = \
    post_2016Q4_cleaned_names

# Drop periods and patch one known alias
yq_total_personnel['PAYEE_CLEANED'] = (
    yq_total_personnel['PAYEE_CLEANED']
    .str.replace('.', '', regex=False).fillna('')
)
yq_total_personnel['PAYEE_CLEANED'] = yq_total_personnel['PAYEE_CLEANED'].replace(
    {'T E ANFINSON': 'THOMAS E ANFINSON'}
)

assert not yq_total_personnel['PAYEE_CLEANED'].str.contains(',').any()
yq_total_personnel.head()


## Relational anomalies

*Maps to manuscript §Exploratory Data Auditing → Relational anomalies.*

Two analyses: the staff co-employment network (Figure 12) and the distribution of staff affiliations across offices.

### Personnel bipartite graph and projection

**Goal.** Build the staff↔member bipartite graph (filtered to staff with at least $1,000 in quarterly compensation) and project it onto the staff side: two staff are linked if they were ever paid by the same member's office.

**Assumptions.** The $1,000 filter excludes one-off small payments (reimbursements, short-term consultants) that would otherwise inflate node counts. Only members with party in {D, R} are kept so the visualisation can use a three-colour partisan palette.

**Interpretation.** `personnel_proj_g` is the staff-to-staff weighted projection. Edge weights are the count of shared employer-quarters. The GEXF export below is consumed by Gephi for the high-resolution publication figure; the inline rendering in the next cell is the trust-contract draft that critics can reproduce from this notebook.

**Validation.** Node/edge counts are printed; the bipartite-graph build asserts no name collision between member-side and staff-side node IDs.


In [ ]:
yq_total_personnel_filtered = yq_total_personnel[
    yq_total_personnel['AMOUNT'] >= 1000
]

personnel_el_df = (
    yq_total_personnel_filtered
    .groupby(['YEAR-QUARTER', 'BIOGUIDE_ID', 'PAYEE_CLEANED'])
    .agg({'AMOUNT': 'sum'}).reset_index()
)
personnel_el_df['member'] = (
    personnel_el_df['BIOGUIDE_ID'].map(member_names_map).str.upper()
)
personnel_el_df.columns = ['yearquarter', 'id', 'payee', 'amount', 'member']

personnel_el_df[['gender', 'party']] = personnel_el_df['id'].apply(
    lambda x: pd.Series(bioguide_gender_party_map.get(x))
)
personnel_el_df = personnel_el_df.dropna(subset=['member'])

filtered_personnel_el = personnel_el_df[personnel_el_df['party'].isin(['D', 'R'])]


In [ ]:
personnel_bp_g = nx.from_pandas_edgelist(
    df=filtered_personnel_el,
    source='member', target='payee', edge_attr=['amount'],
)
nx.write_gexf(personnel_bp_g, 'personnel_bp.gexf')

print(f'Bipartite graph: {personnel_bp_g.number_of_nodes():,} nodes, '
      f'{personnel_bp_g.number_of_edges():,} edges')


In [ ]:
personnel_proj_g = nx.bipartite.projection.weighted_projected_graph(
    personnel_bp_g, filtered_personnel_el['payee'].unique()
)
nx.write_gexf(personnel_proj_g, 'personnel_proj.gexf')

print(f'Projection (staff-to-staff): {personnel_proj_g.number_of_nodes():,} nodes, '
      f'{personnel_proj_g.number_of_edges():,} edges')


### Staff co-employment network (Figure 12)

**Goal.** Render the staff-to-staff co-employment network with a party-only colour encoding (Democratic / Republican / Bipartisan) at landscape aspect ratio. The figure substantiates the manuscript's claim that staff circulation rarely crosses party lines.

**Assumptions.** Only staff with weighted degree ≥ 10 in the projection are drawn — below that, the layout becomes unreadable in print. Bipartisan staff are defined as those whose career touched at least one Democratic and at least one Republican office. The spring-layout seed is fixed.

**Interpretation.** The network is near-bipartite: two large components with few bridging (purple) nodes between them. This visually confirms party-driven staff circulation; the few bipartisan nodes are the cases the manuscript highlights as worth individual inspection.

**Validation.** Node/edge counts of the rendered subgraph are printed; the random seed (42) is fixed so re-execution reproduces the same layout.


In [ ]:
# Each staff member's party history
staff_party = {}
for payee, group in filtered_personnel_el.groupby('payee'):
    has_d = (group['party'] == 'D').any()
    has_r = (group['party'] == 'R').any()
    if has_d and has_r:
        staff_party[payee] = 'B'
    elif has_d:
        staff_party[payee] = 'D'
    elif has_r:
        staff_party[payee] = 'R'

color_map = {'D': PALETTE['D'], 'R': PALETTE['R'], 'B': PALETTE['B']}

# Keep only staff with non-trivial weighted degree for readability
node_strength = dict(personnel_proj_g.degree(weight='weight'))
viz_nodes = [n for n, s in node_strength.items() if s >= 10]
viz_subgraph = personnel_proj_g.subgraph(viz_nodes).copy()
viz_subgraph.remove_nodes_from(list(nx.isolates(viz_subgraph)))

print(f'Visualised subgraph: {viz_subgraph.number_of_nodes():,} nodes, '
      f'{viz_subgraph.number_of_edges():,} edges')

pos = nx.spring_layout(viz_subgraph, k=0.15, seed=42, iterations=50)
node_colors = [
    color_map.get(staff_party.get(n), '#999999') for n in viz_subgraph.nodes()
]


In [ ]:
f, ax = plt.subplots(figsize=(15, 9.5))
nx.draw_networkx_edges(
    viz_subgraph, pos, ax=ax,
    alpha=0.06, width=0.3, edge_color='#888888',
)
nx.draw_networkx_nodes(
    viz_subgraph, pos, ax=ax,
    node_color=node_colors, node_size=18,
    alpha=0.88, linewidths=0,
)
ax.axis('off')
ax.set_title('Staff co-employment network, projected onto staff',
             fontsize=13, loc='left')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PALETTE['D'],
           markersize=11, label='Democratic offices only'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PALETTE['R'],
           markersize=11, label='Republican offices only'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PALETTE['B'],
           markersize=11, label='Bipartisan (both parties)'),
]
ax.legend(handles=legend_elements, loc='upper right',
          fontsize=11, frameon=False)

f.savefig('staff_network.png', dpi=250, facecolor='white')


### Distribution of staff affiliations across offices

**Goal.** Quantify how many distinct member offices each staffer ever drew compensation from. The manuscript characterises the right tail (staff paid by 50+ offices: caucus directors and shared-resource staff).

**Assumptions.** Same $1,000 threshold as the network construction. Distinct offices are counted by Bioguide ID, not by office name.

**Interpretation.** The distribution is heavily right-skewed with median 1 and minimum 1, so mean ± SD would mislead. The cell reports median, IQR, max, and tail-share statistics instead.

**Validation.** Counts are derived from `all_bio_payee_el_df.groupby('payee')['member'].nunique()` — a direct aggregation, no smoothing or imputation.


In [ ]:
# Build the staff-by-office aggregation used by the affiliation summary
all_bio_payee_el_df = (
    yq_total_personnel_filtered
    .groupby(['BIOGUIDE_ID', 'PAYEE_CLEANED'])
    .agg({'AMOUNT': 'sum'})
    .reset_index()
)
all_bio_payee_el_df.columns = ['id', 'payee', 'amount']


In [ ]:
affil_counts = all_bio_payee_el_df.groupby('payee')['id'].nunique()

print(f'Distinct member offices per staffer:')
print(f'  N staffers:                          {len(affil_counts):,}')
print(f'  Minimum:                             {int(affil_counts.min())}')
print(f'  Median:                              {affil_counts.median():.0f}')
print(f'  Interquartile range:                 ['
      f'{int(affil_counts.quantile(0.25))}, '
      f'{int(affil_counts.quantile(0.75))}]')
print(f'  Mean (skew-sensitive):               {affil_counts.mean():.2f}')
print(f'  Maximum:                             {int(affil_counts.max())}')
print(f'  Share with exactly 1 office:         {(affil_counts == 1).mean() * 100:.1f}%')
print(f'  Share with >= 5  offices:            {(affil_counts >= 5).mean() * 100:.1f}%')
print(f'  Share with >= 10 offices:            {(affil_counts >= 10).mean() * 100:.1f}%')
print(f'  Share with >= 50 offices:            {(affil_counts >= 50).mean() * 100:.1f}%')

# Top affiliated staffers
affil_counts.sort_values(ascending=False).head(10)


## Spatial anomalies

*Maps to manuscript §Exploratory Data Auditing → Spatial anomalies.*

One analysis: the linear model of average annual per-member travel spending against geodesic distance from Washington, D.C. (Figure 13). The residuals panel motivates the manuscript's discussion of nonlinearity at long distances.

### Travel spending vs. distance from D.C. (Figure 13)

**Goal.** Fit a linear OLS model of per-member travel spending on geodesic distance from D.C. across state and territory delegations, 2011–2022. Identify delegations whose observed spending sits well above or below the line.

**Assumptions.** Distance is the geodesic kilometre count from each centroid to D.C. (computed in the data-loading section). Spending is aggregated to state-year and divided by the count of distinct members in the delegation to give a per-member figure. State-year is the unit of observation.

**Interpretation.** Left panel: the OLS fit on linear-linear axes, consistent with the linear model. Right panel: residuals — the funnel pattern at long distances motivates the manuscript's discussion of nonlinearity for the most far-flung delegations.

**Validation.** The model summary (slope, intercept, R²) is printed and pinned to named variables. The top 20 absolute residuals and top 20 percentage residuals are computed in `travel_dist_yhat_merged` for inspection.


In [ ]:
members_bioguide_travel_df = members_bioguide_df[
    members_bioguide_df['CATEGORY'] == 'TRAVEL'
]

annual_member_travel = (
    members_bioguide_travel_df
    .groupby(['state', 'YEAR'])
    .agg({'AMOUNT': 'sum', 'BIOGUIDE_ID': 'nunique'})
)
annual_member_travel['per_member'] = (
    annual_member_travel['AMOUNT'] / annual_member_travel['BIOGUIDE_ID']
)
annual_member_travel_per_member = annual_member_travel['per_member'].unstack('YEAR')

avg_travel_spending = annual_member_travel_per_member.stack().reset_index()
avg_travel_spending.columns = ['state', 'year', 'amount']

_merged = pd.merge(
    left=cop_df, right=avg_travel_spending,
    left_on='STUSAB', right_on='state', how='outer',
)

_merged = _merged.dropna()
_merged = _merged[_merged['DC_DIST'] > 0]


In [ ]:
dist_travel_reg = stats.linregress(_merged['DC_DIST'], _merged['amount'])
print(f'OLS travel ~ distance:')
print(f'  slope:     {dist_travel_reg.slope:.2f} USD per km')
print(f'  intercept: {dist_travel_reg.intercept:,.0f} USD')
print(f'  R²:        {dist_travel_reg.rvalue ** 2:.3f}')
print(f'  p:         {dist_travel_reg.pvalue:.2e}')

state_yhats = {
    _state: dist_travel_reg.slope * _dist + dist_travel_reg.intercept
    for _state, _dist in cop_df[['STATE_NAME', 'DC_DIST']].values
}
travel_dist_yhat_df = pd.DataFrame(
    data=state_yhats.values(), index=state_yhats.keys(), columns=['yhat']
)

travel_dist_yhat_merged = pd.merge(
    left=_merged, right=travel_dist_yhat_df,
    left_on='STATE_NAME', right_index=True, how='left',
)
travel_dist_yhat_merged['residual'] = (
    travel_dist_yhat_merged['amount'] - travel_dist_yhat_merged['yhat']
)
travel_dist_yhat_merged['residual_pct'] = (
    travel_dist_yhat_merged['residual'] / travel_dist_yhat_merged['amount']
)


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(13.5, 5.2))

# Panel A: linear-linear model fit
ax.scatter(
    _merged['DC_DIST'], _merged['amount'],
    alpha=0.55, color=PALETTE['bar'], s=22,
    edgecolor='white', linewidth=0.3,
    label='State / territory × year',
)
_x = np.linspace(_merged['DC_DIST'].min(), _merged['DC_DIST'].max(), 100)
ax.plot(
    _x, dist_travel_reg.slope * _x + dist_travel_reg.intercept,
    color=PALETTE['benford'], lw=2.0,
    label=(f'OLS: y = {dist_travel_reg.slope:.2f}·x + '
           f'{dist_travel_reg.intercept:,.0f}\nR² = '
           f'{dist_travel_reg.rvalue ** 2:.3f}'),
)
ax.set_xlabel('Distance from D.C. (km)')
ax.set_ylabel('Avg. annual per-member travel (USD)')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_title('Linear model on linear axes', loc='left')
ax.legend(loc='upper left', fontsize=9)
ax.set_xscale('log')
ax.set_xlim((1e1,2e4))

# Panel B: residuals
# ax = axs[1]
# ax.scatter(
#     _merged['DC_DIST'], travel_dist_yhat_merged['residual'],
#     alpha=0.55, color=PALETTE['bar'], s=22,
#     edgecolor='white', linewidth=0.3,
# )
# ax.axhline(0, color=PALETTE['benford'], lw=1, ls='--')
# ax.set_xlabel('Distance from D.C. (km)')
# ax.set_ylabel('Residual: observed − predicted (USD)')
# ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
# ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
# ax.set_title('Residuals — funnel pattern at long distances', loc='left')

f.suptitle('Travel spending vs. distance from Washington, D.C., 2011–2022',
           fontsize=12, y=1.005)
f.savefig('distance_travel.png')


Top absolute residuals and top percentage residuals — the rows the manuscript discusses (American Samoa, Guam, Northern Mariana Islands, Alaska; Hawaii, Delaware, Maryland).

In [ ]:
print('Top 10 absolute residuals (over-spending vs prediction):')
print(travel_dist_yhat_merged.loc[
    travel_dist_yhat_merged['residual'].abs().nlargest(10).index,
    ['STATE_NAME', 'year', 'amount', 'yhat', 'residual']
].to_string(index=False))

print('\nTop 10 percentage residuals (under-spending where %|residual| largest):')
print(travel_dist_yhat_merged.loc[
    travel_dist_yhat_merged['residual_pct'].abs().nlargest(10).index,
    ['STATE_NAME', 'year', 'amount', 'yhat', 'residual_pct']
].to_string(index=False))
